#Домашнее задание 2

## 1. Загружаем данные

In [18]:
import pandas as pd
import numpy as np
df = pd.read_csv("train.csv")

##2. Основная информация

In [19]:
print("ИНФОРМАЦИЯ О ДАТАСЕТЕ ")
print("\nТипы данных:")
print(df.dtypes)
print("\nПропуски:")
print(df.isnull().sum())
print("\nСредние значения (числовые столбцы):")
print(df.mean(numeric_only=True))
print("\nОписательная статистика:")
print(df.describe())

ИНФОРМАЦИЯ О ДАТАСЕТЕ 

Типы данных:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Пропуски:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Средние значения (числовые столбцы):
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64

Описательная статистика:
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000  

## 3. Процент выживаемости по классам

In [20]:
print("\n ВЫЖИВАЕМОСТЬ ПО КЛАССАМ ")
survival_by_class = df.groupby('Pclass')['Survived'].mean() * 100
print(survival_by_class)


 ВЫЖИВАЕМОСТЬ ПО КЛАССАМ 
Pclass
1    62.962963
2    47.282609
3    24.236253
Name: Survived, dtype: float64


## 4. Самые популярные имена

In [21]:
df['First_Name'] = df['Name'].str.extract(r',\s+(.*?)\s+')
df['Title'] = df['Name'].str.extract(r',\s+(.*?)\.')

#### Мужские имена

In [22]:
def extract_first_name(name): # У меня изначально выводились обращения, поэтому пришлось использовать функцию для разделения "имени" по точке (чтобы взять вторую часть)
    try:
        after_comma = name.split(',')[1]
        after_dot = after_comma.split('.')[1]
        first_name = after_dot.strip().split()[0]
        return first_name
    except:
        return name
df['First_Name'] = df['Name'].apply(extract_first_name)
male_names = df[df['Sex'] == 'male']['First_Name'].value_counts()
print("ТОП-5 мужских имён:")
print(male_names.head(5))
print(f"Самое популярное мужское имя: {male_names.index[0]} ({male_names.iloc[0]} раз)")

ТОП-5 мужских имён:
First_Name
William    35
John       25
George     14
Thomas     13
Charles    13
Name: count, dtype: int64
Самые популярные мужские имя: William (35 раз)


#### Женские имена

In [35]:
import re # встретилась проблема с данными, что некоторые женские имена прописаны после мужских, но в скобочках, предполагаю, что сначала человек который купил, а потом фактический владелец (подарок, например, или служанки/гувернантки/и тп, с мужскими такого не заметила)
def extract_name(name, sex):
    if sex == 'female' and '(' in name and ')' in name:
        # Берём то, что в скобках - это имя женщины
        bracket_match = re.search(r'\((.*?)\)', name)
        if bracket_match:
            # Берём первое слово из скобок
            return bracket_match.group(1).strip().split()[0]
    match = re.search(r',\s+[A-Za-z]+\.\s+([A-Za-z]+)', name)
    if match:
        return match.group(1)
    parts = name.replace('.', ' ').split()
    for i, part in enumerate(parts):
        if part in ['Mr', 'Mrs', 'Miss', 'Ms', 'Master', 'Dr', 'Rev']:
            if i + 1 < len(parts):
                return parts[i + 1]
    return "Unknown"

df['First_Name'] = df.apply(lambda row: extract_name(row['Name'], row['Sex']), axis=1)
female_names = df[df['Sex'] == 'female']['First_Name'].value_counts()
print("\nТОП-5 ЖЕНСКИХ ИМЁН:")
print(female_names.head(5))
print(f"Самое популярное женское имя: {female_names.index[0]} ({female_names.iloc[0]} раз)")


ТОП-5 ЖЕНСКИХ ИМЁН:
First_Name
Anna         15
Mary         14
Elizabeth    11
Margaret     10
Alice         6
Name: count, dtype: int64
Самое популярное женское имя: Anna (15 раз)


## 5. Самые популярные имена по классам

In [44]:
print("ПОПУЛЯРНЫЕ ИМЕНА ПО КЛАССАМ:")
for pclass in sorted(df['Pclass'].unique()):
    print(f"\nКЛАСС {pclass}:")
    male_class = df[(df['Pclass'] == pclass) & (df['Sex'] == 'male')]['First_Name'].value_counts()
    if not male_class.empty:
        print(f"  Мужское: {male_class.index[0]} ({male_class.iloc[0]} раз)")
    female_class = df[(df['Pclass'] == pclass) & (df['Sex'] == 'female')]['First_Name'].value_counts()
    if not female_class.empty:
        print(f"  Женское: {female_class.index[0]} ({female_class.iloc[0]} раз)")

ПОПУЛЯРНЫЕ ИМЕНА ПО КЛАССАМ:

КЛАСС 1:
  Мужское: William (11 раз)
  Женское: Elizabeth (5 раз)

КЛАСС 2:
  Мужское: William (9 раз)
  Женское: Elizabeth (5 раз)

КЛАСС 3:
  Мужское: William (15 раз)
  Женское: Anna (9 раз)


## 6. Пассажиры старше 44 лет

In [25]:
print("\n ПАССАЖИРЫ СТАРШЕ 44 ЛЕТ ")
older_44 = df[df['Age'] > 44]
print(older_44.head(10))


 ПАССАЖИРЫ СТАРШЕ 44 ЛЕТ 
    PassengerId  Survived  Pclass                                      Name  \
6             7         0       1                   McCarthy, Mr. Timothy J   
11           12         1       1                  Bonnell, Miss. Elizabeth   
15           16         1       2          Hewlett, Mrs. (Mary D Kingcome)    
33           34         0       2                     Wheadon, Mr. Edward H   
52           53         1       1  Harper, Mrs. Henry Sleeper (Myna Haxtun)   
54           55         0       1            Ostby, Mr. Engelhart Cornelius   
62           63         0       1               Harris, Mr. Henry Birkhardt   
92           93         0       1               Chaffee, Mr. Herbert Fuller   
94           95         0       3                         Coxon, Mr. Daniel   
96           97         0       1                 Goldschmidt, Mr. George B   

       Sex   Age  SibSp  Parch       Ticket     Fare Cabin Embarked  \
6     male  54.0      0      0  

## 7. Мужчины младше 44 лет

In [26]:
print("\n МУЖЧИНЫ МЛАДШЕ 44 ЛЕТ")
male_under_44 = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
print(male_under_44.head(10))


 МУЖЧИНЫ МЛАДШЕ 44 ЛЕТ
    PassengerId  Survived  Pclass                            Name   Sex   Age  \
0             1         0       3         Braund, Mr. Owen Harris  male  22.0   
4             5         0       3        Allen, Mr. William Henry  male  35.0   
7             8         0       3  Palsson, Master. Gosta Leonard  male   2.0   
12           13         0       3  Saundercock, Mr. William Henry  male  20.0   
13           14         0       3     Andersson, Mr. Anders Johan  male  39.0   
16           17         0       3            Rice, Master. Eugene  male   2.0   
20           21         0       2            Fynney, Mr. Joseph J  male  35.0   
21           22         1       2           Beesley, Mr. Lawrence  male  34.0   
23           24         1       1    Sloper, Mr. William Thompson  male  28.0   
27           28         0       1  Fortune, Mr. Charles Alexander  male  19.0   

    SibSp  Parch     Ticket     Fare        Cabin Embarked First_Name   Title  
0   

## 8. Количество n-местных кабин

In [27]:
print("\n РАСПРЕДЕЛЕНИЕ КАБИН ")
df['Cabin_Type'] = df['Cabin'].fillna('Unknown')
cabin_counts = df['Cabin_Type'].value_counts()
print("Количество пассажиров в каждой каюте:")
print(cabin_counts)


=== РАСПРЕДЕЛЕНИЕ КАБИН ===
Количество пассажиров в каждой каюте:
Cabin_Type
Unknown        687
G6               4
C23 C25 C27      4
B96 B98          4
F2               3
              ... 
E17              1
A24              1
C50              1
B42              1
C148             1
Name: count, Length: 148, dtype: int64


## 9. Пассажиры без родственников

In [28]:
print("\n ПАССАЖИРЫ БЕЗ РОДСТВЕННИКОВ ")
no_relatives = df[(df['SibSp'] == 0) & (df['Parch'] == 0)]
print(f"Количество пассажиров без родственников: {len(no_relatives)}")
print(no_relatives[['Name', 'Age', 'Sex', 'SibSp', 'Parch']].head(10))


 ПАССАЖИРЫ БЕЗ РОДСТВЕННИКОВ 
Количество пассажиров без родственников: 537
                                    Name   Age     Sex  SibSp  Parch
2                 Heikkinen, Miss. Laina  26.0  female      0      0
4               Allen, Mr. William Henry  35.0    male      0      0
5                       Moran, Mr. James   NaN    male      0      0
6                McCarthy, Mr. Timothy J  54.0    male      0      0
11              Bonnell, Miss. Elizabeth  58.0  female      0      0
12        Saundercock, Mr. William Henry  20.0    male      0      0
14  Vestrom, Miss. Hulda Amanda Adolfina  14.0  female      0      0
15      Hewlett, Mrs. (Mary D Kingcome)   55.0  female      0      0
17          Williams, Mr. Charles Eugene   NaN    male      0      0
19               Masselmani, Mrs. Fatima   NaN  female      0      0
